# 01 — Data Exploration

Exploratory data analysis of the crypto return universe: return distributions, correlation structure, volume patterns, and volatility regimes.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Auto-generate synthetic data if none exists
from statarb.utils import load_config
from statarb.data.storage import DataStore
from experiments.generate_synthetic_data import generate_all

cfg = load_config("../experiments/config.yaml")
data_cfg = cfg.get("data", {})
syn_cfg = cfg.get("synthetic", {})
exchange = data_cfg.get("exchange", "binance")
base_dir = data_cfg.get("base_dir", "../data")

store = DataStore(base_dir=base_dir)
symbols = store.list_symbols(exchange, "1d")
if not symbols:
    print("Generating synthetic data...")
    generate_all(
        n_assets=syn_cfg.get("n_assets", 20),
        n_days=syn_cfg.get("n_days", 1000),
        start_date=syn_cfg.get("start_date", "2021-01-01"),
        seed=syn_cfg.get("seed", 42),
        exchange=exchange,
        base_dir=base_dir,
    )
    symbols = store.list_symbols(exchange, "1d")

prices  = store.build_panel(symbols, exchange, "1d", field="close")
volume  = store.build_volume_panel(symbols, exchange, "1d")

from statarb.data.features import FeatureEngine
fe = FeatureEngine()
returns = fe.log_returns(prices)
vol_ratio = fe.volume_ma_ratio(volume, window=21)

print(f"Loaded: {len(symbols)} assets x {len(prices)} days")
print(f"Date range: {prices.index[0].date()} → {prices.index[-1].date()}")


## Return Distributions

In [ ]:
# Daily return summary statistics
stats = returns.describe().T
stats["skew"] = returns.skew()
stats["kurt"] = returns.kurtosis()
stats["ann_vol"] = returns.std() * np.sqrt(365)
print(stats[["mean", "std", "skew", "kurt", "ann_vol", "min", "max"]].round(4).to_string())


In [ ]:
# Histogram of cross-asset mean daily return distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
returns.stack().hist(bins=100, ax=axes[0], color="steelblue", alpha=0.7, edgecolor="none")
axes[0].set_title("Return Distribution (all assets pooled)", fontweight="bold")
axes[0].set_xlabel("Daily Return")
axes[0].set_ylabel("Frequency")
axes[0].axvline(0, color="red", linewidth=1)

# QQ-plot vs normal
from scipy import stats
clean = returns.stack().dropna()
(osm, osr), (slope, intercept, r) = stats.probplot(clean, dist="norm")
axes[1].scatter(osm, osr, s=1, alpha=0.3, color="steelblue")
axes[1].plot(osm, slope * np.array(osm) + intercept, color="red", linewidth=1.5)
axes[1].set_title("QQ-Plot vs Normal", fontweight="bold")
axes[1].set_xlabel("Theoretical Quantiles")
axes[1].set_ylabel("Sample Quantiles")
plt.tight_layout()
plt.show()
print(f"Excess kurtosis (pooled): {clean.kurtosis():.2f}")


## Correlation Structure

In [ ]:
# Rolling correlation between all pairs
corr = returns.dropna().corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Pairwise Return Correlation Matrix", fontweight="bold")
plt.tight_layout()
plt.show()

# Distribution of pairwise correlations
pairs = corr.values[np.triu_indices_from(corr.values, k=1)]
print(f"Median pairwise correlation: {np.median(pairs):.3f}")
print(f"Mean pairwise correlation: {np.mean(pairs):.3f}")


In [ ]:
# Rolling median correlation (market stress indicator)
def rolling_median_corr(rets, window=63):
    result = []
    for i in range(window, len(rets)):
        window_rets = rets.iloc[i - window:i]
        c = window_rets.corr().values
        vals = c[np.triu_indices_from(c, k=1)]
        result.append(np.median(vals))
    return pd.Series(result, index=rets.index[window:])

roll_corr = rolling_median_corr(returns.dropna(axis=1, how="all").fillna(0))
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(roll_corr.index, roll_corr.values, linewidth=1.2, color="steelblue")
ax.fill_between(roll_corr.index, roll_corr.values, alpha=0.2, color="steelblue")
ax.axhline(float(roll_corr.mean()), color="red", linestyle="--",
           linewidth=1, label=f"Mean={roll_corr.mean():.2f}")
ax.set_title("Rolling Median Pairwise Correlation (63-day)", fontweight="bold")
ax.set_ylabel("Median Correlation")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## Volume Patterns

In [ ]:
# Volume normalized by its rolling average (activity ratio)
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Mean activity ratio across universe
avg_activity = vol_ratio.mean(axis=1)
axes[0].plot(avg_activity.index, avg_activity.values, linewidth=0.8, color="darkorange")
axes[0].axhline(1.0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_title("Average Volume / 21-day MA (universe mean)", fontweight="bold")
axes[0].set_ylabel("V/MA Ratio")

# Day-of-week volume pattern
dow = pd.DataFrame({
    "volume_ratio": avg_activity.values,
    "dayofweek":    avg_activity.index.dayofweek,
}).dropna()
dow_avg = dow.groupby("dayofweek")["volume_ratio"].mean()
labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
axes[1].bar(range(len(dow_avg)), dow_avg.values, color="steelblue", alpha=0.8)
axes[1].set_xticks(range(len(dow_avg)))
axes[1].set_xticklabels(labels[:len(dow_avg)])
axes[1].set_title("Volume by Day of Week", fontweight="bold")
axes[1].set_ylabel("Avg V/MA Ratio")
axes[1].axhline(1.0, color="black", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.show()


## Volatility Regimes

In [ ]:
# Cross-sectional median realized volatility
realized_vol = fe.realized_volatility(returns, window=21, annualize=True, periods_per_year=365)
median_vol = realized_vol.median(axis=1)

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(median_vol.index, median_vol.values * 100, alpha=0.4, color="crimson")
ax.plot(median_vol.index, median_vol.values * 100, color="crimson", linewidth=0.8)
ax.set_title("Median Annualized Realized Volatility (21-day, universe)", fontweight="bold")
ax.set_ylabel("Annualized Vol (%)")
ax.axhline(float(median_vol.mean()) * 100, color="black", linestyle="--",
           linewidth=1, label=f"Mean={median_vol.mean()*100:.0f}%")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

# High-vol regime = days when median vol > 1.5x its 63-day average
vol_ma = median_vol.rolling(63).mean()
high_vol_regime = (median_vol > 1.5 * vol_ma)
print(f"High-vol regime: {high_vol_regime.mean()*100:.1f}% of days")
print(f"Mean vol in high-vol regime: {median_vol[high_vol_regime].mean()*100:.0f}%")
print(f"Mean vol in low-vol regime:  {median_vol[~high_vol_regime].mean()*100:.0f}%")


In [ ]:
# Cross-sectional dispersion (key driver of L/S alpha)
cs_vol = returns.std(axis=1)
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(cs_vol.index, cs_vol.values * 100, linewidth=0.8, color="navy")
ax.set_title("Cross-Sectional Return Dispersion (daily std, all assets)", fontweight="bold")
ax.set_ylabel("Daily CS Std (%)")
plt.tight_layout()
plt.show()
print(f"Average cross-sectional dispersion: {cs_vol.mean()*100:.2f}% per day")
print(f"This determines the maximum signal IC → alpha ceiling.")
